In [1]:
import pandas

/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('Detailed_Polling_Data.csv')

<IPython.core.display.Javascript object>

In [3]:
df.columns

Index(['serial no. of polling station',
       'all india anna dravida munnetra kazhagam', 'bahujan samaj party',
       'naam tamilar katchi', 'dravida munnetra kazhagam', 'samata party',
       'makkal nalvaazhvuk katchi', 'tamizhaga vaazhvurimai katchi',
       'veerath thiyagi viswanathadoss thozhilalarkal katchi',
       'desiya makkal sakthi katchi', 'republican party of india (athawale)',
       'tamilaga vettri kazhagam', 'independent', 'independent.1',
       'independent.2', 'independent.3', 'independent.4', 'independent.5',
       'independent.6', 'independent.7', 'independent.8', 'independent.9',
       'independent.10', 'independent.11', 'independent.12', 'independent.13',
       'independent.14', 'independent.15', 'independent.16', 'independent.17',
       'independent.18', 'independent.19', 'independent.20', 'independent.21',
       'independent.22', 'independent.23', 'total of valid votes',
       'no. of rejected votes', 'nota', 'total', 'no. of tendered votes',
      

In [5]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the third dataset (Update the filename to match your file)
df3 = pd.read_csv("Detailed_Polling_Data.csv")

# 2. Select core political parties present in this constituency
core_parties = [
    'all india anna dravida munnetra kazhagam', 
    'dravida munnetra kazhagam', 
    'naam tamilar katchi', 
    'tamilaga vettri kazhagam'
]

# Fill missing vote counts with 0
df3[core_parties] = df3[core_parties].fillna(0)

# 3. Calculate true total votes for normalization (Core Parties + Independents + NOTA)
df3['Total_Calculated_Votes'] = df3[core_parties].sum(axis=1) + df3['Total_Independent_Votes'].fillna(0) + df3['nota'].fillna(0)

# Filter out empty booths to prevent division by zero
df3 = df3[df3['Total_Calculated_Votes'] > 0].copy()

# 4. Feature Engineering: Generate exact vote shares (%)
share_cols = []
for party in core_parties:
    col_name = f'{party}_share_pct'
    df3[col_name] = (df3[party] / df3['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

# Add Independent Share and Margin Percentage as structural features
df3['independent_share_pct'] = (df3['Total_Independent_Votes'].fillna(0) / df3['Total_Calculated_Votes']) * 100
feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']

# Fill any remaining NaNs in features
df3[feature_cols] = df3[feature_cols].fillna(0)

# 5. Extract and scale features for the ML model
X = df3[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df3['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Raw Profile Breakdown to help identify what each cluster means
print("\n--- DATASET 3: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df3.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 3: BOOTH COUNT PER CLUSTER ---")
print(df3['Cluster_ID'].value_counts())

# 8. Export individual target files for campaign ground teams
for cluster_num in range(optimal_k):
    cluster_df = df3[df3['Cluster_ID'] == cluster_num][
        [
            'serial no. of polling station', 
            'location and name of building in which polling station located', 
            'polling area', 
            'Winner_Party', 
            'Margin_Percentage'
        ]
    ]
    filename = f"Dataset_3_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Generated 4 cluster files based on the new party dynamics.")



--- DATASET 3: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            all india anna dravida munnetra kazhagam_share_pct  \
Cluster_ID                                                       
0                                                       10.28    
1                                                       12.61    
2                                                        4.98    
3                                                        8.09    

            dravida munnetra kazhagam_share_pct  \
Cluster_ID                                        
0                                         31.92   
1                                         39.76   
2                                         57.20   
3                                         42.57   

            naam tamilar katchi_share_pct  tamilaga vettri kazhagam_share_pct  \
Cluster_ID                                                                      
0                                    2.91                               5